# Data import from Vexcel

In [18]:
# import modules
import requests
from pyproj import Transformer
import leafmap.foliumap as leafmap
from localtileserver import TileClient
import rasterio 
from pathlib import Path
import os
from dotenv import load_dotenv

In [19]:
# prepare download directories
## get project root directory
project_dir = Path.cwd()

## if notebook in notebooks folder, move one level up to root directory
if project_dir.name in ["notebooks", "code"]:
    project_dir = project_dir.parent

## create download directory in data folder
download_dir = project_dir / "data" / "raw" / "raster" / "vexcel"
download_dir.mkdir(parents=True, exist_ok=True)

print(download_dir.resolve())

C:\Users\Maja\OneDrive - Universität Zürich UZH\1-Module\HS26\sds320\project\data\raw\raster\vexcel


In [20]:
#define bounds and transforme them
bounds_LV95 = (2666000, 1210000, 2667000, 1211000)
transformer = Transformer.from_crs("EPSG:2056", "EPSG:4326", always_xy=True)
bounds_WGS84 = transformer.transform_bounds(bounds_LV95[0], bounds_LV95[1], bounds_LV95[2], bounds_LV95[3])# transform crs

# Als WKT-Polygon formatieren(nötig für VEXCEL api abfrage):
min_lon, min_lat, max_lon, max_lat = bounds_WGS84[0], bounds_WGS84[1], bounds_WGS84[2], bounds_WGS84[3]
wkt_bounds = f"POLYGON(({bounds_WGS84[0]} {min_lat}, {max_lon} {min_lat}, {max_lon} {max_lat}, {min_lon} {max_lat}, {min_lon} {min_lat}))"
print(min_lon, min_lat, max_lon, max_lat)

8.307034437925985 47.03765288164638 8.320338769540836 47.04674761991993


In [21]:
# define token (externalized in an environment file that is not being committed to git)
# Copy token again if next code cell shows code 403, should work again afterwards

load_dotenv("../.env", override=True)

api_token = os.getenv("VEXCEL_API_TOKEN")

if not api_token:
    raise ValueError("Vexcel API token was not loaded")

headers = {"Authorization": api_token}

print("Token loaded:", api_token is not None)
print("Starts with Bearer:", api_token.startswith("Bearer ") if api_token else False)
print("Has surrounding whitespace:", api_token != api_token.strip() if api_token else None)
print("Authorization header exists:", "Authorization" in headers)

Token loaded: True
Starts with Bearer: True
Has surrounding whitespace: False
Authorization header exists: True


In [22]:
# define api and parameters
api_url = "https://api.vexcelgroup.com/v2/ortho/collections"
params = {"limit": 100, "wkt" : wkt_bounds, "srid" : 4326, 
           "bands" : "irg"}


#get and list available collections 
r = requests.get(api_url, headers= headers, params = params)
if r.status_code == 200:
    data = r.json()
    for feature in data.get("features", []):
            props = feature.get("properties", {})
            print(f"- Collection: {props.get('collection')}, type: {props.get('product-type')}")
else: 
    print(r.status_code)
    print(r.text)


- Collection: ch-lu-luzern-2025, type: final-ortho
- Collection: n47e9-de-bw-engen-2024, type: final-ortho
- Collection: n47e9-de-bw-engen-2022, type: final-ortho
- Collection: n46e9-ch-nw-stans-2022, type: final-ortho


In [23]:
#plot outlines
m = leafmap.Map(height="300px")
m.add_basemap("OpenStreetMap")
m.add_geojson(data)

if data.get("features"):
    m.add_geojson(
        data, 
        layer_name="Vexcel Coverage",
        fields=["collection"]  # Fields shown in tooltip/popup
    )

m

In [25]:
#extract rgb GeoTIFF
api_url = "https://api.vexcelgroup.com/v2/ortho/extract"
params = {"wkt" : wkt_bounds, "srid" : 4326, 
          "collection" : "ch-lu-luzern-2025", 
          "image-format" : "tiff" }

r = requests.get(api_url, headers= headers, params = params)
if r.status_code == 200: 
    filepath = download_dir / "extracted_ortho_rgb_2025.tif"
    with open(filepath, "wb") as f:
        f.write(r.content)
    print("generating map")
    m1 = leafmap.Map(height="300px")
    m1.add_basemap("OpenStreetMap")

    # use tile client becuse url is alwasy broken
    tile_client = TileClient(filepath, host="127.0.0.1")
    m1.add_raster(tile_client)
else:
    print(r.status_code)

m1


generating map


In [26]:
#download cir (contains NIR)
api_url = "https://api.vexcelgroup.com/v2/ortho/extract"
params = {"wkt" : wkt_bounds, "srid" : 4326, 
          "collection" : "ch-lu-luzern-2025", 
          "image-format" : "tiff", 
           "bands" : "irg" }

r = requests.get(api_url, headers= headers, params = params)
if r.status_code == 200: 
    filepath = download_dir / "extracted_ortho_irg_2025.tif"
    with open(filepath, "wb") as f:
        f.write(r.content)
    print("generating map")
    m3 = leafmap.Map(height="600px")
    m3.add_basemap("OpenStreetMap")

    # use tile client becuse url is alwasy broken
    tile_client = TileClient(filepath, host="127.0.0.1")
    m3.add_raster(tile_client)
    
else:
    print(r.status_code)

m3


generating map


In [13]:
help(download_dir)

Help on WindowsPath in module pathlib object:

class WindowsPath(Path, PureWindowsPath)
 |  WindowsPath(*args, **kwargs)
 |
 |  Path subclass for Windows systems.
 |
 |  On a Windows system, instantiating a Path should return this object.
 |
 |  Method resolution order:
 |      WindowsPath
 |      Path
 |      PureWindowsPath
 |      PurePath
 |      builtins.object
 |
 |  Methods inherited from Path:
 |
 |  __enter__(self)
 |
 |  __exit__(self, t, v, tb)
 |
 |  __init__(self, *args, **kwargs)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |
 |  absolute(self)
 |      Return an absolute version of this path by prepending the current
 |      working directory. No normalization or symlink resolution is performed.
 |
 |      Use resolve() to get the canonical path to a file.
 |
 |  chmod(self, mode, *, follow_symlinks=True)
 |      Change the permissions of the path, like os.chmod().
 |
 |  exists(self, *, follow_symlinks=True)
 |      Whether this path exists.
 |

## Inspect Metadata of extracted orthophotos
- Path: where file is stored (both tif files should be in notebooks\data\vexcel)
*-> maybe change path afterwards to have clean structure ? (to \data\raw\raster)*
- Wdith x height: columns x  rows
- Bands: RGB (irg file is false colour image)
- CRS: here in Web Mercator coordinate system
- Resolution: in m, here about 30cm resolution 
*-> check again if this should really be like that or shouldn't it be 7.5cm? Check in Vexcel Data Viewer !*
- Bounds: bounding box of tile
- Data type: uint8 = each band's pixel values are unsigned 8-bit integers

In [28]:
def inspect_raster(path):
    with rasterio.open(path) as src:
        print("Path:", path)
        print("Width x height:", src.width, "x", src.height)
        print("Bands:", src.count)
        print("CRS:", src.crs)
        print("Resolution:", src.res)
        print("Bounds:", src.bounds)
        print("Data type:", src.dtypes[0])


inspect_raster(download_dir / "extracted_ortho_irg_2025.tif")
print("\n")
inspect_raster(download_dir / "extracted_ortho_rgb_2025.tif")

Path: c:\Users\Maja\OneDrive - Universität Zürich UZH\1-Module\HS26\sds320\project\data\raw\raster\vexcel\extracted_ortho_irg_2025.tif
Width x height: 4962 x 4977
Bands: 3
CRS: EPSG:3857
Resolution: (0.2985821417389747, 0.298582141738971)
Bounds: BoundingBox(left=924734.5710297788, bottom=5948222.051581389, right=926216.1356170876, top=5949708.094900824)
Data type: uint8


Path: c:\Users\Maja\OneDrive - Universität Zürich UZH\1-Module\HS26\sds320\project\data\raw\raster\vexcel\extracted_ortho_rgb_2025.tif
Width x height: 4962 x 4977
Bands: 3
CRS: EPSG:3857
Resolution: (0.2985821417389747, 0.298582141738971)
Bounds: BoundingBox(left=924734.5710297788, bottom=5948222.051581389, right=926216.1356170876, top=5949708.094900824)
Data type: uint8


In [29]:
m2 = leafmap.Map(height = 600)

m2.split_map(
    left_layer = "extracted_ortho_rgb_2025.tif",
    right_layer = "extracted_ortho_irg_2025.tif"
)

display(m2)

Exception in callback _ProactorBasePipeTransport._call_connection_lost(None)
handle: <Handle _ProactorBasePipeTransport._call_connection_lost(None)>
Traceback (most recent call last):
  File "c:\Users\Maja\programming\anaconda3\envs\sds320\Lib\asyncio\events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
  File "c:\Users\Maja\programming\anaconda3\envs\sds320\Lib\asyncio\proactor_events.py", line 165, in _call_connection_lost
    self._sock.shutdown(socket.SHUT_RDWR)
ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host
Exception in callback _ProactorBasePipeTransport._call_connection_lost(None)
handle: <Handle _ProactorBasePipeTransport._call_connection_lost(None)>
Traceback (most recent call last):
  File "c:\Users\Maja\programming\anaconda3\envs\sds320\Lib\asyncio\events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
  File "c:\Users\Maja\programming\anaconda3\envs\sds320\Lib\asyncio\p